In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.2.1 requires msgspec, which is not installed.
unsloth-zoo 2026.2.1 requires tyro, which is not installed.
unsloth-zoo 2026.2.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.0.0 which is incompatible.
unsloth-zoo 2026.2.1 requires torchao>=0.13.0, but you have torchao 0.10.0 which is incompatible.
unsloth-zoo 2026.2.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0

In [ ]:
# Import Libraries for Finetuning
from unsloth import FastLanguageModel, is_bf16_supported # for my t4 gpu
from datasets import load_dataset
from trl import SFTTrainer # transformer Reinforcement Learning
from transformers import TrainingArguments # hyperparameters for training


max_seq_length = 512  # 1024 for 8b model

model ,tokenizer = FastLanguageModel.from_pretrained(model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
                                    max_seq_length=max_seq_length,
                                    dtype=None,
                                    load_in_4bit=True)

# LoRA (Low Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank of the low-rank matrices
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, # Scaling factor
    lora_dropout = 0, # We are not using dropout to get more training speed
    bias = "none", # Bias type
    use_gradient_checkpointing = "unsloth", # it is unsloth custom memory saver
)

# We will prepare our Dataset (with manual stopping token)

def format_prompts(examples):
    prompt_template = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
                        {problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
                        {answer}<|eot_id|>"""
    inputs = [msg[0]['content'] for msg in examples['messages']]
    outputs = examples['reannotated_assistant_content']

    texts = []
    for problem, answer in zip(inputs, outputs):
        text = prompt_template.format(problem=problem, answer=answer)
        texts.append(text)

    # we will use tokenizer to truncate the texts to max_seq_length
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=max_seq_length,
        add_special_tokens=False # Our template already has special tokens
    )

    # We will return the actual tokeinzed IDs instead of texts
    return {"input_ids": tokenized["input_ids"], "attention_mask": tokenized["attention_mask"]}

# We will load dataset
dataset = load_dataset("ServiceNow-AI/R1-Distill-SFT", "v1" ,split="train[:5000]")

# We will apply the format_prompts function to the dataset
dataset = dataset.map(format_prompts, batched=True, remove_columns=dataset.column_names)

# We will define Training Arguments
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=60, # if you want better loss optimization the use more max steps
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit", # it is unsloth custom memory saver
        seed=3407,
        output_dir="lora_outputs",
    )
)

trainer.train()

# We will save the model
model.save_pretrained("my_llama3_Model")
tokenizer.save_pretrained("my_llama3_Model_tokenizer")




🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/1679162 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.139739
2,1.134696
3,1.149402
4,1.134890
5,1.023622
6,1.042352
7,0.974119
8,0.919718
9,0.859245
10,0.818155


('my_llama3_Model_tokenizer/tokenizer_config.json',
 'my_llama3_Model_tokenizer/chat_template.jinja',
 'my_llama3_Model_tokenizer/tokenizer.json')

In [ ]:
import torch
# Testing
# We will define our prompt template for testing
prompt_style = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
                        {problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

# question = "if i have 3 shirt and it takes 3 hours to dry them outside, how long will it take to dry 30 shirts?"
question = "harsh has 3 sisters, each of his sisters has 2 brothers, how many brothers does harsh have?"
# We will format the question
prompt = prompt_style.format(problem=question)

# We will tokenize the question
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# We will generate the answer
outputs = model.generate(**inputs, max_new_tokens=256)

# We will decode the answer
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("Answer:", answer)

Question: harsh has 3 sisters, each of his sisters has 2 brothers, how many brothers does harsh have?
Answer: user
                        harsh has 3 sisters, each of his sisters has 2 brothers, how many brothers does harsh have?assistant
                        <think>
First, I need to understand the problem: Harsh has three sisters, and each of his sisters has two brothers. The question asks how many brothers Harsh has.

Let's break this down step by step:

1. **Harsh's sisters:** Harsh has three sisters, so there are three of them.

2. **Each sister has two brothers:** This means each of Harsh's three sisters has two brothers. So, each sister has two brothers in total.

3. **Adding up the brothers:** Since each of Harsh's three sisters has two brothers, the total number of brothers is:
   - Sister 1 has 2 brothers.
   - Sister 2 has 2 brothers.
   - Sister 3 has 2 brothers.

Adding these up:
   - 2 + 2 + 2 = 6

4. **Conclusion:** Therefore, Harsh has **6 brothers**.
</think>

To de